<a href="https://colab.research.google.com/github/thhh07/Analizador-l-xico-Python-Colab/blob/main/Compiladores_prj_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Programa 01 devera traduzir as expressões infixas com

In [ ]:
# ==========================================
# PROGRAMA 1 - INFIXA PARA PÓS-FIXA
# Disciplina: Compiladores
# ==========================================

# Função que define a prioridade dos operadores
def precedencia(operador):

    if operador == '+' or operador == '-':
        return 1

    if operador == '*' or operador == '/':
        return 2

    return 0


# Função principal de conversão
def infixa_para_posfixa(expressao):

    pilha = []
    saida = []

    i = 0

    while i < len(expressao):

        caractere = expressao[i]

        # Ignorar espaços
        if caractere == ' ':
            i += 1
            continue

        # Reconhecer números inteiros
        if caractere.isdigit():

            numero = ''

            while i < len(expressao) and expressao[i].isdigit():
                numero += expressao[i]
                i += 1

            saida.append(numero)

            continue

        # Reconhecer operadores
        elif caractere in ['+', '-', '*', '/']:

            while (len(pilha) > 0 and
                   precedencia(pilha[-1]) >= precedencia(caractere)):

                saida.append(pilha.pop())

            pilha.append(caractere)

        i += 1

    # Desempilhar operadores restantes
    while len(pilha) > 0:
        saida.append(pilha.pop())

    return ' '.join(saida)


# ==========================================
# PROGRAMA PRINCIPAL
# ==========================================

expressao = input("Digite a expressão infixa: ")

resultado = infixa_para_posfixa(expressao)

print("\nExpressão Pós-Fixa:")
print(resultado)

Digite a expressão infixa: 1 +2 * 3

Expressão Pós-Fixa:
1 2 3 * +


PROGRAMA 2/3 - ANALISADOR LÉXICO

In [ ]:
class LexicalAnalyzer:
    def __init__(self, source_code):
        self.code = source_code
        self.pos = 0       # Posição atual no código
        self.line = 1      # Linha atual

        # 1. Inicializar a tabela de símbolos com mais de uma palavra-chave
        self.symbol_table = {
            'if': 'KEYWORD',
            'else': 'KEYWORD',
            'while': 'KEYWORD',
            'int': 'KEYWORD',
            'float': 'KEYWORD',
            'return': 'KEYWORD'
        }
        self.tokens = []
        self.errors = []

    def current_char(self):
        """Retorna o caractere atual."""
        if self.pos < len(self.code):
            return self.code[self.pos]
        return None

    def peek(self):
        """Olha o próximo caractere sem avançar a posição."""
        if self.pos + 1 < len(self.code):
            return self.code[self.pos + 1]
        return None

    def advance(self):
        """Avança para o próximo caractere e controla a contagem de linhas."""
        if self.pos < len(self.code):
            if self.code[self.pos] == '\n':
                self.line += 1
            self.pos += 1

    def analyze(self):
        """Executa a análise léxica."""
        while self.pos < len(self.code):
            char = self.current_char()

            # 6. Remover espaços em branco, antes e depois de qualquer lexema
            if char.isspace():
                self.advance()
                continue

            # 7. Remover os dois tipos de comentários e 3. Reconhecer divisão
            if char == '/':
                next_char = self.peek()
                if next_char == '/':
                    # Comentário de uma linha (//)
                    self.advance() # consome primeiro '/'
                    self.advance() # consome segundo '/'
                    while self.current_char() is not None and self.current_char() != '\n':
                        self.advance()
                    continue
                elif next_char == '*':
                    # Comentário de múltiplas linhas (/* ... */)
                    self.advance() # consome '/'
                    self.advance() # consome '*'
                    while self.current_char() is not None:
                        if self.current_char() == '*' and self.peek() == '/':
                            self.advance() # consome '*'
                            self.advance() # consome '/'
                            break
                        self.advance()
                    continue
                else:
                    # Se não for comentário, é o operador de divisão
                    self.tokens.append(('ARITHMETIC_OP', '/'))
                    self.advance()
                    continue

            # 3. Reconhecer os outros três operadores aritméticos (+, -, *)
            if char in ['+', '-', '*']:
                self.tokens.append(('ARITHMETIC_OP', char))
                self.advance()
                continue

            # 4. Reconhecer operadores relacionais (==, !=, <, >, <=, >=) e atribuição (=)
            if char in ['=', '!', '<', '>']:
                next_char = self.peek()
                if next_char == '=':
                    self.tokens.append(('RELATIONAL_OP', char + '='))
                    self.advance() # consome o operador inicial
                    self.advance() # consome o '='
                else:
                    if char == '=':
                        self.tokens.append(('ASSIGNMENT_OP', '=')) # Atribuição
                        self.advance()
                    elif char == '!':
                        # Um '!' sozinho não é um operador relacional no nosso contexto
                        self.errors.append((self.line, f"Erro Léxico: Caractere inesperado '!' (esperava-se '!=')"))
                        self.advance()
                    else:
                        self.tokens.append(('RELATIONAL_OP', char)) # '<' ou '>'
                        self.advance()
                continue

            # 2. Reconhecer números decimais (e inteiros)
            if char.isdigit():
                num_str = ''
                has_dot = False
                while self.current_char() is not None and (self.current_char().isdigit() or self.current_char() == '.'):
                    if self.current_char() == '.':
                        if has_dot:
                            break # Evita dois pontos num mesmo número (ex: 3.14.15)
                        has_dot = True
                    num_str += self.current_char()
                    self.advance()
                self.tokens.append(('DECIMAL_NUM' if has_dot else 'INTEGER_NUM', num_str))
                continue

            # 5. Reconhecer identificadores e armazená-los na tabela de símbolos
            if char.isalpha() or char == '_':
                id_str = ''
                while self.current_char() is not None and (self.current_char().isalnum() or self.current_char() == '_'):
                    id_str += self.current_char()
                    self.advance()

                # Verifica se já está na tabela de símbolos (ex: é palavra-chave ou já foi lido)
                if id_str in self.symbol_table:
                    token_type = self.symbol_table[id_str]
                    self.tokens.append((token_type, id_str))
                else:
                    # Se não existe, armazena na tabela de símbolos
                    self.symbol_table[id_str] = 'IDENTIFIER'
                    self.tokens.append(('IDENTIFIER', id_str))
                continue

            # 8 e 9. Tratar erros, gravar mensagem/linha, e RECUPERAR de uma posição
            self.errors.append((self.line, f"Erro Léxico: Caractere não válido '{char}'"))
            self.advance()

        return self.tokens, self.errors, self.symbol_table


# =====================================================================
# INTERFACE DE ENTRADA DIGITADA (Para rodar no Colab)
# =====================================================================
print("=== ANALISADOR LÉXICO INTERATIVO ===")
print("Digite ou cole o seu código abaixo.")
print("Quando terminar, pressione ENTER duas vezes seguidas (deixe uma linha em branco) para executar:\n")

linhas_codigo = []
while True:
    linha = input()
    if linha == "":  # Se a linha for vazia, encerra a digitação
        break
    linhas_codigo.append(linha)

# Junta todas as linhas digitadas em uma única string separada por quebras de linha
codigo_fonte_digitado = "\n".join(linhas_codigo)

if not codigo_fonte_digitado.strip():
    print("\n[Aviso] Nenhum código foi digitado. Encerrando execução.")
else:
    print("\n" + "="*50)
    print("Iniciando Processamento da Entrada...")
    print("="*50 + "\n")

    # Executa o analisador com a entrada que você digitou
    lexer = LexicalAnalyzer(codigo_fonte_digitado)
    tokens_gerados, erros_encontrados, tabela_simbolos = lexer.analyze()

    print("-" * 40)
    print("TOKENS RECONHECIDOS:")
    print("-" * 40)
    for t in tokens_gerados:
        print(f"[{t[0]}] -> {t[1]}")

    print("\n" + "-" * 40)
    print("ERROS ENCONTRADOS (Com recuperação de 1 posição):")
    print("-" * 40)
    if not erros_encontrados:
        print("Nenhum erro encontrado de sintaxe léxica!")
    else:
        for err in erros_encontrados:
            print(f"Linha {err[0]}: {err[1]}")

    print("\n" + "-" * 40)
    print("TABELA DE SÍMBOLOS FINAL:")
    print("-" * 40)
    for chave, valor in tabela_simbolos.items():
        print(f"'{chave}': {valor}")




=== ANALISADOR LÉXICO INTERATIVO ===
Digite ou cole o seu código abaixo.
Quando terminar, pressione ENTER duas vezes seguidas (deixe uma linha em branco) para executar:

int valor = 45.20; while (valor > 10) @  valor = valor - 1;


Iniciando Processamento da Entrada...

----------------------------------------
TOKENS RECONHECIDOS:
----------------------------------------
[KEYWORD] -> int
[IDENTIFIER] -> valor
[ASSIGNMENT_OP] -> =
[DECIMAL_NUM] -> 45.20
[KEYWORD] -> while
[IDENTIFIER] -> valor
[RELATIONAL_OP] -> >
[INTEGER_NUM] -> 10
[IDENTIFIER] -> valor
[ASSIGNMENT_OP] -> =
[IDENTIFIER] -> valor
[ARITHMETIC_OP] -> -
[INTEGER_NUM] -> 1

----------------------------------------
ERROS ENCONTRADOS (Com recuperação de 1 posição):
----------------------------------------
Linha 1: Erro Léxico: Caractere não válido ';'
Linha 1: Erro Léxico: Caractere não válido '('
Linha 1: Erro Léxico: Caractere não válido ')'
Linha 1: Erro Léxico: Caractere não válido '@'
Linha 1: Erro Léxico: Caractere não

Entrada

int valor = 45.20;
while (valor > 10) @
valor = valor - 1;

In [ ]:
import re

class AnalisadorLexico:
    def __init__(self):
        # 1. Definição das Palavras-Chave da nossa linguagem fictícia
        self.palavras_chave = {'if', 'else', 'while', 'for', 'return', 'int', 'float', 'void', 'print'}

        # 2. Definição das regras de expressões regulares (Regex) para cada tipo de Token
        # A ordem aqui importa: padrões mais específicos devem vir antes para evitar conflitos.
        regras_tokens = [
            ('COMENTARIO_MULTI', r'/\*.*?\*/'),        # Comentários de múltiplas linhas: /* ... */
            ('COMENTARIO_LINHA', r'//.*'),             # Comentários de uma linha: // ...
            ('NUM_FLOAT',        r'\d+\.\d+'),         # Números decimais (ex: 3.14)
            ('NUM_INT',          r'\d+'),              # Números inteiros (ex: 42)
            ('TEXTO',            r'".*?"|\'.*?\''),    # Strings entre aspas duplas ou simples
            ('IDENTIFICADOR',    r'[A-Za-z_][A-Za-z0-9_]*'), # Nomes de variáveis e funções
            ('OPERADOR',         r'==|!=|<=|>=|[+\-*/=<>]'), # Operadores matemáticos e lógicos
            ('PONTUACAO',        r'[(){}\[\];,.]'),    # Símbolos delimitadores
            ('QUEBRA_LINHA',     r'\n'),               # Quebra de linha (usado para contar as linhas)
            ('ESPACO',           r'[ \t]+'),           # Espaços e tabulações (serão ignorados)
            ('ERRO',             r'.'),                # Qualquer outro caractere não mapeado cai aqui
        ]

        # Junta todas as regras em uma única Regex gigante, nomeando os grupos
        self.regex_mestre = '|'.join(f'(?P<{nome}>{padrao})' for nome, padrao in regras_tokens)

        # Compila a Regex (re.DOTALL permite que o .* do comentário multilinhas pegue quebras de linha)
        self.compilador = re.compile(self.regex_mestre, re.DOTALL)

    def analisar(self, codigo_fonte):
        tokens_encontrados = []
        erros_encontrados = []
        linha_atual = 1

        # Posição do último caractere de quebra de linha (ajuda a calcular a coluna)
        inicio_linha_pos = 0

        # finditer varre o texto e encontra todas as correspondências baseadas nas nossas regras
        for match in self.compilador.finditer(codigo_fonte):
            tipo = match.lastgroup
            valor = match.group()
            coluna = match.start() - inicio_linha_pos

            if tipo == 'QUEBRA_LINHA':
                inicio_linha_pos = match.end()
                linha_atual += 1
                continue

            elif tipo == 'ESPACO':
                continue # Apenas ignora os espaços em branco

            elif tipo in ['COMENTARIO_LINHA', 'COMENTARIO_MULTI']:
                # Se o comentário tiver quebras de linha dentro dele (/* \n */), atualizamos o contador
                quebras_no_comentario = valor.count('\n')
                if quebras_no_comentario > 0:
                    linha_atual += quebras_no_comentario
                    # Ajusta o início da linha para a última quebra encontrada dentro do comentário
                    inicio_linha_pos = match.start() + valor.rfind('\n') + 1
                continue # Removemos os comentários simplesmente não adicionando-os à lista de tokens

            elif tipo == 'ERRO':
                erro_msg = f"Linha {linha_atual}, Coluna {coluna}: Caractere não reconhecido '{valor}'"
                erros_encontrados.append(erro_msg)
                continue

            elif tipo == 'IDENTIFICADOR' and valor in self.palavras_chave:
                tipo = 'PALAVRA_CHAVE'

            # Adiciona o token válido na lista
            tokens_encontrados.append((tipo, valor, linha_atual, coluna))

        return tokens_encontrados, erros_encontrados

# ==========================================
# ÁREA INTERATIVA PARA O GOOGLE COLAB
# ==========================================

print("=== Analisador Léxico em Python ===")
print("Digite seu código abaixo. Para finalizar e analisar, digite 'FIM' em uma linha vazia.\n")

linhas_codigo = []
while True:
    linha = input()
    if linha.strip().upper() == 'FIM':
        break
    linhas_codigo.append(linha)

codigo_teste = '\n'.join(linhas_codigo)

# Instancia e roda o analisador
lexico = AnalisadorLexico()
tokens, erros = lexico.analisar(codigo_teste)

print("\n--- RESULTADO DA ANÁLISE ---")

if erros:
    print("\n[!] ERROS LÉXICOS ENCONTRADOS:")
    for erro in erros:
        print(f"  -> {erro}")

print("\n[+] TOKENS RECONHECIDOS:")
print(f"{'TIPO':<17} | {'VALOR':<15} | {'LINHA':<5} | {'COLUNA'}")
print("-" * 55)
for tipo, valor, linha, col in tokens:
    # Formatando a saída para ficar parecida com uma tabela
    print(f"{tipo:<17} | {valor:<15} | {linha:<5} | {col}")

=== Analisador Léxico em Python ===
Digite seu código abaixo. Para finalizar e analisar, digite 'FIM' em uma linha vazia.

int main() {     // Definindo uma variavel     float taxa = 3.14;      /* Comentario de        multiplas linhas */     int @numero_invalido = 10;          if (taxa >= 3.0) {         print("Alto");     } } FIM

fim

--- RESULTADO DA ANÁLISE ---

[+] TOKENS RECONHECIDOS:
TIPO              | VALOR           | LINHA | COLUNA
-------------------------------------------------------
PALAVRA_CHAVE     | int             | 1     | 0
IDENTIFICADOR     | main            | 1     | 4
PONTUACAO         | (               | 1     | 8
PONTUACAO         | )               | 1     | 9
PONTUACAO         | {               | 1     | 11


Programa 3

In [ ]:
import re

class AnalisadorLexico:
    def __init__(self):
        # 1. Inicializar a tabela de símbolos com mais de uma palavra-chave
        self.tabela_simbolos = {
            "if": "PALAVRA_CHAVE",
            "else": "PALAVRA_CHAVE",
            "while": "PALAVRA_CHAVE",
            "int": "PALAVRA_CHAVE",
            "float": "PALAVRA_CHAVE",
            "return": "PALAVRA_CHAVE"
        }
        self.tokens = []
        self.erros = []

    def analisar(self, codigo_fonte):
        self.tokens = []
        self.erros = []

        i = 0
        linha = 1
        n = len(codigo_fonte)

        while i < n:
            ch = codigo_fonte[i]

            # Controle de linhas físicas
            if ch == '\n':
                linha += 1
                i += 1
                continue

            # Remover espaços em branco (antes e depois de qualquer lexema)
            if ch.isspace():
                i += 1
                continue

            # Remover comentários de linha única "//"
            if ch == '/' and i + 1 < n and codigo_fonte[i+1] == '/':
                i += 2
                while i < n and codigo_fonte[i] != '\n':
                    i += 1
                continue

            # Remover comentários multilinha "/* ... */"
            if ch == '/' and i + 1 < n and codigo_fonte[i+1] == '*':
                i += 2
                comentario_fechado = False
                while i < n:
                    if codigo_fonte[i] == '\n':
                        linha += 1
                    if codigo_fonte[i] == '*' and i + 1 < n and codigo_fonte[i+1] == '/':
                        i += 2
                        comentario_fechado = True
                        break
                    i += 1
                if not comentario_fechado:
                    self.erros.append(f"Erro: Comentário multilinha não fechado na linha {linha}")
                continue

            # Reconhecer Ponto e Vírgula (Fim de linha de código)
            if ch == ';':
                self.tokens.append(("DELIMITADOR", ";", linha))
                i += 1
                continue

            # Reconhecer os quatro operadores aritméticos
            if ch in ['+', '-', '*', '/']:
                self.tokens.append(("OP_ARITMETICO", ch, linha))
                i += 1
                continue

            # Reconhecer operadores relacionais e atribuição
            if ch in ['<', '>', '=', '!']:
                if i + 1 < n and codigo_fonte[i+1] == '=':
                    op = ch + '='
                    self.tokens.append(("OP_RELACIONAL", op, linha))
                    i += 2
                else:
                    if ch == '=':
                        self.tokens.append(("OP_ATRIBUICAO", ch, linha))
                    else:
                        self.tokens.append(("OP_RELACIONAL", ch, linha))
                    i += 1
                continue

            # Reconhecer números decimais e inteiros
            if ch.isdigit():
                num_str = ""
                while i < n and codigo_fonte[i].isdigit():
                    num_str += codigo_fonte[i]
                    i += 1

                # Verifica se possui parte decimal
                if i < n and codigo_fonte[i] == '.':
                    num_str += '.'
                    i += 1
                    if i < n and codigo_fonte[i].isdigit():
                        while i < n and codigo_fonte[i].isdigit():
                            num_str += codigo_fonte[i]
                            i += 1
                    else:
                        self.erros.append(f"Erro Lexico: Número decimal malformado na linha {linha}")
                        continue

                self.tokens.append(("NUMERO", num_str, linha))
                continue

            # Reconhecer identificadores e armazená-los na tabela de símbolos
            if ch.isalpha() or ch == '_':
                id_str = ""
                while i < n and (codigo_fonte[i].isalnum() or codigo_fonte[i] == '_'):
                    id_str += codigo_fonte[i]
                    i += 1

                if id_str in self.tabela_simbolos:
                    self.tokens.append((self.tabela_simbolos[id_str], id_str, linha))
                else:
                    # Armazena dinamicamente na Tabela de Símbolos se for novo
                    self.tabela_simbolos[id_str] = "IDENTIFICADOR"
                    self.tokens.append(("IDENTIFICADOR", id_str, sku := linha))
                continue

            # Tratamento e Recuperação de erro de uma posição (ignora o caractere inválido)
            self.erros.append(f"Erro Lexico: Caractere inválido '{ch}' detectado na linha {linha}")
            i += 1

    def exibir_resultados(self):
        print("--- TOKENS RECONHECIDOS ---")
        for token, lexema, linha in self.tokens:
            print(f"Linha {linha} | Token: {token:<15} | Lexeme: {lexema}")

        print("\n--- ERROS ENCONTRADOS ---")
        if not self.erros:
            print("Nenhum erro léxico detectado!")
        else:
            for erro in self.erros:
                print(erro)

        print("\n--- TABELA DE SÍMBOLOS FINAL ---")
        for chave, valor in self.tabela_simbolos.items():
            print(f"Lexema: {chave:<12} -> Tipo: {valor}")

# --- BLOCO DE TESTE NO COLAB ---
codigo_teste = """
int preco = 100;
float desconto = 15.50; // Comentário de linha

/* Comentário
   multilinha */
while (preco > 50) {
    preco = preco - desconto;
}
@ # Caracteres inválidos para testar recuperação de erro
if (preco == 25.5) {
    return 0;
}
"""

analisador = AnalisadorLexico()
analisador.analisar(codigo_teste)
analisador.exibir_resultados()

--- TOKENS RECONHECIDOS ---
Linha 2 | Token: PALAVRA_CHAVE   | Lexeme: int
Linha 2 | Token: IDENTIFICADOR   | Lexeme: preco
Linha 2 | Token: OP_ATRIBUICAO   | Lexeme: =
Linha 2 | Token: NUMERO          | Lexeme: 100
Linha 2 | Token: DELIMITADOR     | Lexeme: ;
Linha 3 | Token: PALAVRA_CHAVE   | Lexeme: float
Linha 3 | Token: IDENTIFICADOR   | Lexeme: desconto
Linha 3 | Token: OP_ATRIBUICAO   | Lexeme: =
Linha 3 | Token: NUMERO          | Lexeme: 15.50
Linha 3 | Token: DELIMITADOR     | Lexeme: ;
Linha 7 | Token: PALAVRA_CHAVE   | Lexeme: while
Linha 7 | Token: IDENTIFICADOR   | Lexeme: preco
Linha 7 | Token: OP_RELACIONAL   | Lexeme: >
Linha 7 | Token: NUMERO          | Lexeme: 50
Linha 8 | Token: IDENTIFICADOR   | Lexeme: preco
Linha 8 | Token: OP_ATRIBUICAO   | Lexeme: =
Linha 8 | Token: IDENTIFICADOR   | Lexeme: preco
Linha 8 | Token: OP_ARITMETICO   | Lexeme: -
Linha 8 | Token: IDENTIFICADOR   | Lexeme: desconto
Linha 8 | Token: DELIMITADOR     | Lexeme: ;
Linha 10 | Token: IDENTIF